# CFA Study Buddy — AI Chatbot for CFA Candidates

**Final Project — LLM-Based Tools and Gemini API Integration for Data Scientists**

**Author:** Reinaldi Santoso

---

## 1. Project Overview

**Project Name:** CFA Study Buddy

**Target Pengguna:**
CFA candidates (Level I, II, dan III) yang sedang mempersiapkan exam. Cocok untuk:
- Self-study candidates yang tidak ikut prep course mahal
- Working professionals yang butuh quick clarification di sela jam kerja
- Mahasiswa S1/S2 di Indonesia yang baru kenal CFA curriculum
- Study group facilitator yang butuh second opinion saat diskusi konsep

**Bagaimana Chatbot Membantu Pengguna:**

Chatbot ini berperan sebagai personal tutor 24/7 yang fokus pada CFA curriculum di seluruh 10 topik utama: Quantitative Methods, Economics, Financial Statement Analysis, Corporate Issuers, Equity Investments, Fixed Income, Derivatives, Alternative Investments, Portfolio Management, dan Ethics & Professional Standards. Pengguna dapat menanyakan penjelasan konsep, formula derivation, perbedaan antar metode, contoh kalkulasi, hingga tips menjawab item set questions. Chatbot dibekali system prompt yang menempatkan dirinya sebagai CFA charterholder mentor — bukan generic AI — sehingga responsnya terstruktur, menggunakan terminologi CFA Institute yang konsisten, dan selalu mengaitkan jawaban ke Learning Outcome Statements (LOS) yang relevan.

**Konfigurasi Parameter Kreatif:**

| Parameter | Opsi | Fungsi |
|---|---|---|
| **Tone gaya bahasa** | Formal / Educational / Casual | Menyesuaikan kebutuhan: formal untuk reading material, edukatif untuk explanation, casual untuk brainstorming |
| **Bahasa output** | English / Bahasa Indonesia | Dual-language untuk membantu candidate Indonesia memahami konsep dalam bahasa native sebelum exam (yang akan dijawab dalam English) |
| **CFA Level focus** | Level I / II / III / All | Mengarahkan depth dan complexity jawaban sesuai exam level user |
| **Temperature** | 0.0 – 1.0 (slider) | Mengontrol kreativitas vs. konsistensi jawaban. Rendah untuk formula & definisi, tinggi untuk diskusi konseptual |
| **Max output tokens** | 256 – 4096 (slider) | Membatasi panjang respons sesuai konteks penggunaan |
| **Memory percakapan** | Persistent via session_state | Bot mengingat konteks (misalnya saat user follow-up: "jelaskan lebih dalam soal yang tadi") |
| **Reset percakapan** | Tombol di sidebar | Hapus history untuk topik baru tanpa harus refresh app |
| **Export chat history** | Download .txt / .md | User dapat menyimpan diskusi sebagai catatan belajar |

---

## 2. Architecture Overview

```
User Browser
    ↕  (HTTPS)
ngrok tunnel  ←→  Google Colab (port 8501)
                           ↕
                   Streamlit App
                           ↕
                   Google Gemini API
                  (gemini-2.5-flash)
```

**Stack:**
- **Frontend & Logic:** Streamlit (st.chat_message, st.chat_input, st.session_state)
- **LLM:** Google Gemini 2.5 Flash via `google-genai` SDK
- **Hosting (development):** Google Colab + ngrok tunnel
- **Memory:** Streamlit session_state + Gemini chat session (server-side context)

---

## 3. Setup — Install Library & Konfigurasi ngrok

### Step 1 — Install dependencies

In [8]:
!pip install -q streamlit pyngrok google-genai

### Step 2 — Set ngrok auth token

Cara mendapatkan token:
1. Daftar gratis di [https://ngrok.com](https://ngrok.com)
2. Login → klik **Your Authtoken** di dashboard
3. Copy token, paste ke **Colab Secrets** dengan nama `NGROK_TOKEN` (klik ikon 🔑 di sidebar kiri Colab)

In [9]:
from pyngrok import ngrok
from google.colab import userdata

ngrok.set_auth_token(userdata.get('NGROK_TOKEN'))
print("ngrok token configured successfully.")

ngrok token configured successfully.


### Step 3 — Helper function untuk jalankan Streamlit + tunnel ngrok

In [10]:
import subprocess
import time

def run_streamlit(filename, port=8501):
    # Kill all existing streamlit processes
    subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
    # Force-free port jika masih dipakai
    subprocess.run(["fuser", "-k", f"{port}/tcp"], capture_output=True)
    # Tutup semua tunnel ngrok
    ngrok.kill()
    # Tunggu port benar-benar bebas
    time.sleep(3)

    proc = subprocess.Popen(
        [
            "streamlit", "run", filename,
            "--server.headless=true",
            "--server.port", str(port),
            "--server.enableCORS=false",
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )
    time.sleep(3)
    public_url = ngrok.connect(port)
    print(f"Streamlit running at: {public_url}")
    return proc

---

## 4. Build the Chatbot App

Berikut adalah file utama `cfa_study_buddy.py` yang berisi seluruh logic chatbot. Cell ini menulis file ke disk Colab untuk kemudian dijalankan via Streamlit.

In [11]:
%%writefile cfa_study_buddy.py
# =============================================================================
# CFA Study Buddy — AI Chatbot for CFA Candidates
# Author: Reinaldi Santoso
# Stack:  Streamlit + Google Gemini API (gemini-2.5-flash)
# =============================================================================
import streamlit as st
from google import genai
from google.genai import types
from datetime import datetime

# ── 1. PAGE CONFIG ───────────────────────────────────────────────────────────
st.set_page_config(
    page_title="CFA Study Buddy",
    page_icon="📘",
    layout="centered",
)

st.title("📘 CFA Study Buddy")
st.caption("Personal AI tutor for CFA Level I, II, and III candidates — powered by Gemini")

# ── 2. SYSTEM PROMPT TEMPLATES ───────────────────────────────────────────────
# All instruction blocks are written in English. The output language is
# controlled by a dedicated LANGUAGE_INSTRUCTIONS block, placed at the TOP of
# the final prompt with an explicit override rule. This prevents the model
# from being 'pulled' toward the language of nearby instructional text.

TONE_INSTRUCTIONS = {
    "Formal":      "Use a formal, professional tone similar to the CFA Institute curriculum textbook. Avoid humor and casual expressions.",
    "Educational": "Use the tone of a patient lecturer. Provide analogies, numerical examples, and step-by-step breakdowns.",
    "Casual":      "Use a relaxed, conversational tone — like a study partner who is also preparing for the CFA exam. Stay accurate but informal.",
}

LANGUAGE_INSTRUCTIONS = {
    "English": (
        "OUTPUT LANGUAGE: You MUST respond in English ONLY, regardless of "
        "the language the user writes in. If the user writes in Indonesian, "
        "Spanish, or any other language, still respond entirely in English. "
        "Use CFA Institute terminology consistently. "
        "This rule overrides any default behavior to mirror the user's language."
    ),
    "Bahasa Indonesia": (
        "OUTPUT LANGUAGE: You MUST respond in Bahasa Indonesia ONLY, regardless "
        "of the language the user writes in. If the user writes in English or "
        "any other language, still respond entirely in Bahasa Indonesia. "
        "Keep CFA technical terms in English when no precise Indonesian "
        "equivalent exists (examples: 'duration', 'convexity', 'WACC', "
        "'yield to maturity'). This rule overrides any default behavior to "
        "mirror the user's language."
    ),
}

LEVEL_INSTRUCTIONS = {
    "Level I":    "Focus on Level I scope: definitions, basic formulas, and foundational concepts. Avoid Level II/III topics unless the user explicitly asks.",
    "Level II":   "Focus on Level II scope: application of concepts, valuation, and analysis. Assume the user has mastered Level I material.",
    "Level III":  "Focus on Level III scope: portfolio management, wealth planning, and synthesis of prior-level concepts.",
    "All Levels": "Answer based on the question context without restricting to a specific level. Mention which level a topic is taught at when relevant.",
}

BASE_PROMPT = """You are CFA Study Buddy, a dedicated AI tutor for CFA Program candidates.
Your knowledge base covers the CFA Institute curriculum for Levels I, II, and III, including:
Quantitative Methods, Economics, Financial Statement Analysis, Corporate Issuers,
Equity Investments, Fixed Income, Derivatives, Alternative Investments,
Portfolio Management & Wealth Planning, and Ethics & Professional Standards.

Your behavior rules:
1. Reference relevant Learning Outcome Statements (LOS) when appropriate.
2. For formula-based questions, always show the formula first, then explain each term, then provide a numerical example.
3. For Ethics questions, identify which Standard (I-VII) or sub-section applies, then explain the resolution per CFA Institute guidance.
4. When the user asks about something OUTSIDE the CFA curriculum, politely redirect: 'That is outside the CFA curriculum scope. I am here to help you prepare for the exam. Would you like to ask about a related CFA topic instead?'
5. Never guarantee exam outcomes or provide leaked exam content. Always note that your answers reflect the curriculum as commonly taught, not actual exam questions.
6. If you are uncertain about a fact, say so explicitly. Do not fabricate formula values, dates, or CFA Institute policies.
"""

def build_system_prompt(tone: str, language: str, level: str) -> str:
    """Compose final system prompt from user-selected modifiers.

    IMPORTANT: Language instruction is placed FIRST so it is the most
    salient directive for the model. Placing it last (or in the middle)
    causes Gemini to default to mirroring the user's input language,
    especially when other instruction blocks contain mixed-language text.
    """
    return (
        LANGUAGE_INSTRUCTIONS[language]
        + "\n\n" + BASE_PROMPT
        + "\n\nTONE: " + TONE_INSTRUCTIONS[tone]
        + "\n\nEXAM FOCUS: " + LEVEL_INSTRUCTIONS[level]
        + "\n\nFINAL REMINDER: Before sending your response, verify it is "
        + "written in the language specified at the top of these instructions."
    )

# ── 3. SIDEBAR — SETTINGS ────────────────────────────────────────────────────
with st.sidebar:
    st.subheader("🔐 API Configuration")
    google_api_key = st.text_input(
        "Google AI API Key",
        type="password",
        help="Dapatkan gratis di https://aistudio.google.com",
    )

    st.divider()
    st.subheader("🎓 Study Preferences")

    level = st.selectbox(
        "CFA Level Focus",
        ["Level I", "Level II", "Level III", "All Levels"],
        index=0,
        help="Bot akan menyesuaikan depth jawaban dengan level yang dipilih.",
    )

    language = st.selectbox(
        "Response Language",
        ["English", "Bahasa Indonesia"],
        index=0,
    )

    tone = st.selectbox(
        "Tone",
        ["Educational", "Formal", "Casual"],
        index=0,
    )

    st.divider()
    st.subheader("⚙️ Generation Parameters")

    temperature = st.slider(
        "Temperature",
        min_value=0.0,
        max_value=1.0,
        value=0.3,
        step=0.05,
        help="Rendah (0.0–0.3) = konsisten, cocok untuk formula. Tinggi (0.7+) = lebih kreatif, cocok untuk diskusi konseptual.",
    )

    max_tokens = st.slider(
        "Max Output Tokens",
        min_value=256,
        max_value=4096,
        value=1024,
        step=128,
        help="Batas panjang jawaban. 1024 ≈ ~700–800 kata.",
    )

    st.divider()
    st.subheader("🛠️ Session")
    reset_button = st.button("🔄 Reset Conversation", use_container_width=True)

    # Export chat history — hanya muncul kalau ada history
    if st.session_state.get("messages"):
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

        # Build .txt content
        txt_lines = [
            "CFA Study Buddy — Conversation Export",
            f"Exported: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
            f"Level: {level} | Language: {language} | Tone: {tone}",
            "=" * 60, "",
        ]
        for m in st.session_state.messages:
            role = "YOU" if m["role"] == "user" else "CFA STUDY BUDDY"
            txt_lines.append(f"[{role}]")
            txt_lines.append(m["content"])
            txt_lines.append("")
        txt_content = "\n".join(txt_lines)

        # Build .md content
        md_lines = [
            "# CFA Study Buddy — Conversation Export", "",
            f"**Exported:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}  ",
            f"**Level:** {level} | **Language:** {language} | **Tone:** {tone}", "",
            "---", "",
        ]
        for m in st.session_state.messages:
            role = "### 👤 You" if m["role"] == "user" else "### 🤖 CFA Study Buddy"
            md_lines.append(role)
            md_lines.append("")
            md_lines.append(m["content"])
            md_lines.append("")

        md_content = "\n".join(md_lines)

        st.download_button(
            label="📄 Export as .txt",
            data=txt_content,
            file_name=f"cfa_chat_{timestamp}.txt",
            mime="text/plain",
            use_container_width=True,
        )
        st.download_button(
            label="📝 Export as .md",
            data=md_content,
            file_name=f"cfa_chat_{timestamp}.md",
            mime="text/markdown",
            use_container_width=True,
        )

    st.divider()
    st.caption("⚠️ Disclaimer: This tool is for study assistance only. Always verify against official CFA Institute curriculum.")

# ── 4. API KEY GATE ──────────────────────────────────────────────────────────
if not google_api_key:
    st.info("👈 Masukkan Google AI API Key di sidebar untuk mulai belajar.", icon="🗝️")
    st.markdown(
        """
        ### Welcome to CFA Study Buddy

        Personal AI tutor untuk perjalanan CFA Program kamu. Fitur:

        - 📚 **All 10 CFA topics** — Quant, FRA, Equity, FI, Derivatives, dst.
        - 🎯 **Level-specific focus** — Pilih Level I/II/III untuk depth yang tepat
        - 🌐 **Dual language** — English atau Bahasa Indonesia
        - 💾 **Export history** — Simpan diskusi sebagai catatan belajar
        - 🔄 **Conversation memory** — Bot ingat konteks untuk follow-up questions

        **Cara mulai:**
        1. Dapatkan API key gratis di [Google AI Studio](https://aistudio.google.com)
        2. Paste di sidebar
        3. Pilih Level & preferensi kamu
        4. Mulai bertanya!
        """
    )
    st.stop()

# ── 5. INITIALIZE GEMINI CLIENT ──────────────────────────────────────────────
# Build current system prompt + cek apakah ada perubahan key/preferensi.
# Kalau ada perubahan, re-initialize chat session agar config baru terpakai.
current_system_prompt = build_system_prompt(tone, language, level)
current_config_signature = f"{google_api_key}|{tone}|{language}|{level}|{temperature}|{max_tokens}"

if (
    "genai_client" not in st.session_state
    or st.session_state.get("_last_key") != google_api_key
):
    try:
        st.session_state.genai_client = genai.Client(api_key=google_api_key)
        st.session_state._last_key = google_api_key
        # Force re-init chat session karena key berubah
        st.session_state.pop("chat", None)
        st.session_state.pop("messages", None)
    except Exception as e:
        st.error(f"❌ API Key tidak valid: {e}")
        st.stop()

# ── 6. INITIALIZE CHAT SESSION ───────────────────────────────────────────────
# Re-init chat kalau:
# - Belum ada chat session, ATAU
# - Konfigurasi (tone/language/level/temp/max_tokens) berubah
if (
    "chat" not in st.session_state
    or st.session_state.get("_config_signature") != current_config_signature
):
    st.session_state.chat = st.session_state.genai_client.chats.create(
        model="gemini-2.5-flash",
        config=types.GenerateContentConfig(
            system_instruction=current_system_prompt,
            temperature=temperature,
            max_output_tokens=max_tokens,
        ),
    )
    st.session_state._config_signature = current_config_signature

if "messages" not in st.session_state:
    st.session_state.messages = []

# ── 7. RESET BUTTON HANDLER ──────────────────────────────────────────────────
if reset_button:
    st.session_state.pop("chat", None)
    st.session_state.pop("messages", None)
    st.session_state.pop("_config_signature", None)
    st.rerun()

# ── 8. DISPLAY CHAT HISTORY ──────────────────────────────────────────────────
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# ── 9. HANDLE USER INPUT ─────────────────────────────────────────────────────
prompt = st.chat_input("Ask me anything about the CFA curriculum...")

if prompt:
    # Append & render user message
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    # Send to Gemini & render assistant message
    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            try:
                response = st.session_state.chat.send_message(prompt)
                answer = response.text if hasattr(response, "text") else str(response)
            except Exception as e:
                answer = f"⚠️ Terjadi error saat memanggil Gemini API: `{e}`\n\nCoba periksa API key atau kurangi max output tokens."
        st.markdown(answer)

    st.session_state.messages.append({"role": "assistant", "content": answer})


Overwriting cfa_study_buddy.py


---

## 5. Run the App

Cell di bawah ini akan menjalankan Streamlit app dan membuka tunnel ngrok. Setelah menjalankan, klik URL `https://...ngrok-free.app` yang muncul untuk membuka chatbot di browser.

In [13]:
proc = run_streamlit("cfa_study_buddy.py")

Streamlit running at: NgrokTunnel: "https://monotone-duchess-fedora.ngrok-free.dev" -> "http://localhost:8501"


### 5.1 Cara Menggunakan App

1. **Buka URL ngrok** yang muncul di output cell di atas
2. **Paste Google AI API Key** di sidebar (dapatkan gratis di [aistudio.google.com](https://aistudio.google.com))
3. **Konfigurasi study preferences:**
   - **CFA Level Focus** — pilih level yang sedang kamu prep
   - **Language** — English atau Bahasa Indonesia
   - **Tone** — Educational (default), Formal, atau Casual
4. **Sesuaikan generation parameters:**
   - **Temperature** — 0.0–0.3 untuk pertanyaan formula/definisi, 0.5+ untuk diskusi konseptual
   - **Max Output Tokens** — atur sesuai panjang jawaban yang dibutuhkan
5. **Mulai bertanya** di kotak chat input bagian bawah

### 5.2 Contoh Prompt untuk Testing

**Level I — Quantitative Methods:**
> Explain the difference between geometric mean and arithmetic mean. When should I use each one?

**Level I — Fixed Income:**
> Jelaskan apa itu duration dan convexity. Kenapa convexity adalah good thing untuk bond investor?

**Level II — Equity Valuation:**
> Walk me through the Gordon Growth Model. When does it break down?

**Level III — Portfolio Management:**
> What are the key differences between strategic and tactical asset allocation in IPS construction?

**Ethics (semua level):**
> I overheard my colleague at a coffee shop discussing material non-public information about an upcoming earnings release. Which CFA Standard applies, and what should I do?

**Follow-up test (memory):**
> Setelah pertanyaan duration di atas, coba tanya: "Bisa kasih contoh kalkulasi modified duration untuk bond yang barusan kamu jelaskan?" — Bot harus ingat konteks bond sebelumnya.

---

## 6. Stop the App

Jalankan cell di bawah untuk menghentikan Streamlit dan menutup tunnel ngrok ketika sudah selesai.

In [14]:
try:
    proc.terminate()
    print("✅ Streamlit stopped.")
except Exception:
    print("ℹ️ No active Streamlit process.")

ngrok.kill()
print("✅ ngrok tunnels closed.")

✅ Streamlit stopped.
✅ ngrok tunnels closed.


---

## 7. Technical Notes & Design Decisions

### Mengapa system prompt dibangun dinamis?

Pendekatan `build_system_prompt(tone, language, level)` lebih maintainable daripada hard-coding satu prompt panjang. Setiap user preference (tone, language, level) menjadi modular block yang bisa di-toggle independen. Bila ingin menambah preference baru (misalnya "include numerical examples: yes/no"), tinggal tambah dictionary baru dan injection di function builder.

### Mengapa chat session di-recreate saat preferensi berubah?

Gemini SDK menerima `config` (termasuk `system_instruction`, `temperature`, `max_output_tokens`) hanya saat `chats.create()`. Setelah session terbentuk, config tersebut tidak bisa diubah mid-conversation. Solusinya: track `config_signature` di session state, dan bila berubah, re-initialize chat session. Trade-off: conversation history di sisi Gemini akan hilang saat preferensi berganti — tapi history di UI tetap (disimpan di `st.session_state.messages`).

### Mengapa temperature default 0.3?

CFA candidates lebih butuh konsistensi & akurasi daripada kreativitas. Temperature rendah (0.3) memberikan jawaban yang lebih deterministik untuk konsep, formula, dan definisi — yang sebagian besar adalah pertanyaan CFA. User dapat menaikkan slider untuk pertanyaan open-ended seperti diskusi konseptual atau brainstorming.

### Limitasi yang dijelaskan jujur ke user

System prompt secara eksplisit menginstruksikan bot untuk:
- Tidak fabricate formula values atau CFA Institute policies
- Mengakui ketidakpastian ("if you are uncertain, say so")
- Redirect pertanyaan di luar scope CFA curriculum
- Tidak menjanjikan hasil exam atau membahas leaked content

Ini selaras dengan responsible AI practice — bot adalah study aid, bukan pengganti official curriculum.

---

## 8. Deliverables Checklist

- [x] **AI Chatbot dengan use case spesifik** — CFA exam preparation tutor
- [x] **Konfigurasi parameter kreatif** — tone, language, CFA level, temperature, max tokens, memory, export
- [x] **NLP/LLM untuk natural language processing** — Google Gemini 2.5 Flash
- [x] **URL repositori GitHub** — lihat README.md
- [x] **Screenshots UI** — terlampir di repo (`/screenshots`)

---

## 9. References

- [Streamlit Documentation](https://docs.streamlit.io)
- [Google Gen AI Python SDK](https://googleapis.github.io/python-genai/)
- [Google AI Studio](https://aistudio.google.com)
- [pyngrok](https://pyngrok.readthedocs.io/)
- [CFA Institute Program Curriculum](https://www.cfainstitute.org/programs/cfa)